# Iris Dataset Exploration with Apache Spark

このNotebookでは、Apache Sparkを使用してIrisデータセットを探索し、機械学習モデルを構築します。

In [ ]:
// Spark 依存関係の読み込み（Scala 2.13 を明示的に指定）
import $ivy.`org.apache.spark:spark-sql_2.13:3.5.0`
import $ivy.`org.apache.spark:spark-mllib_2.13:3.5.0`

println("Spark 依存関係が正常にロードされました")

## 1. 環境設定とライブラリのインポート

In [1]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.ml.{Pipeline, PipelineModel}
import org.apache.spark.ml.classification.DecisionTreeClassifier
import org.apache.spark.ml.feature.{StringIndexer, VectorAssembler}
import org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator

// SparkSessionの作成
val spark = SparkSession.builder()
  .appName("IrisExploration")
  .master("local[*]")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

println("Spark Session created successfully!")
println(s"Spark version: ${spark.version}")

-- [E008] Not Found Error: cmd2.sc:1:11 ----------------------------------------
1 |import org.apache.spark.sql.SparkSession
  |       ^^^^^^^^^^
  |       value apache is not a member of org
-- [E008] Not Found Error: cmd2.sc:2:11 ----------------------------------------
2 |import org.apache.spark.ml.{Pipeline, PipelineModel}
  |       ^^^^^^^^^^
  |       value apache is not a member of org
-- [E008] Not Found Error: cmd2.sc:3:11 ----------------------------------------
3 |import org.apache.spark.ml.classification.DecisionTreeClassifier
  |       ^^^^^^^^^^
  |       value apache is not a member of org
-- [E008] Not Found Error: cmd2.sc:4:11 ----------------------------------------
4 |import org.apache.spark.ml.feature.{StringIndexer, VectorAssembler}
  |       ^^^^^^^^^^
  |       value apache is not a member of org
-- [E008] Not Found Error: cmd2.sc:5:11 ----------------------------------------
5 |import org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator
  |       ^^^

## 2. データの読み込み

In [1]:
// データの読み込み
val df = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("../data/iris.csv")

println(s"データ件数: ${df.count()}")
println("\nスキーマ:")
df.printSchema()

-- [E006] Not Found Error: cmd2.sc:1:9 -----------------------------------------
1 |val df = spark.read
  |         ^^^^^
  |         Not found: spark
  |
  | longer explanation available when compiling with `-explain`
Compilation Failed

## 3. データの概要確認

In [ ]:
// 最初の5行を表示
df.show(5, truncate = false)

In [ ]:
// 統計情報
df.describe("sepal_length", "sepal_width", "petal_length", "petal_width").show()

In [ ]:
// 種類ごとの件数
df.groupBy("species").count().show()

## 4. 特徴量の準備

In [ ]:
// ラベルのインデックス化
val labelIndexer = new StringIndexer()
  .setInputCol("species")
  .setOutputCol("label")

// 特徴量のベクトル化
val assembler = new VectorAssembler()
  .setInputCols(Array("sepal_length", "sepal_width", "petal_length", "petal_width"))
  .setOutputCol("features")
  .setHandleInvalid("skip")

// パイプラインで変換
val prepPipeline = new Pipeline().setStages(Array(labelIndexer, assembler))
val preparedDf = prepPipeline.fit(df).transform(df)

println(s"準備後のデータ件数: ${preparedDf.count()}")
preparedDf.select("features", "label", "species").show(5, truncate = false)

## 5. データの分割

In [ ]:
// 訓練データとテストデータに分割
val Array(trainData, testData) = preparedDf.randomSplit(Array(0.7, 0.3), seed = 42)

println(s"訓練データ: ${trainData.count()} 件")
println(s"テストデータ: ${testData.count()} 件")

## 6. モデルの訓練

In [ ]:
// Decision Treeモデルの作成
val dt = new DecisionTreeClassifier()
  .setLabelCol("label")
  .setFeaturesCol("features")
  .setMaxDepth(5)

val pipeline = new Pipeline().setStages(Array(dt))

// モデルの訓練
println("モデルを訓練中...")
val model = pipeline.fit(trainData)
println("訓練完了！")

## 7. モデルの評価

In [ ]:
// テストデータで予測
val predictions = model.transform(testData)

// 評価メトリクスの計算
val evaluator = new MulticlassClassificationEvaluator()
  .setLabelCol("label")
  .setPredictionCol("prediction")

val accuracy = evaluator.setMetricName("accuracy").evaluate(predictions)
val precision = evaluator.setMetricName("weightedPrecision").evaluate(predictions)
val recall = evaluator.setMetricName("weightedRecall").evaluate(predictions)
val f1 = evaluator.setMetricName("f1").evaluate(predictions)

println(f"Accuracy: ${accuracy * 100}%.2f%%")
println(f"Precision: ${precision * 100}%.2f%%")
println(f"Recall: ${recall * 100}%.2f%%")
println(f"F1 Score: ${f1 * 100}%.2f%%")

## 8. 予測結果の確認

In [ ]:
// 予測結果のサンプル表示
predictions.select(
  "sepal_length", "sepal_width", "petal_length", "petal_width",
  "species", "label", "prediction"
).show(10, truncate = false)

In [ ]:
// 予測の正誤を確認
val correct = predictions.filter("label == prediction").count()
val total = predictions.count()

println(s"正解: $correct / $total")
println(f"正解率: ${correct.toDouble / total * 100}%.2f%%")

## 9. Decision Treeの構造確認

In [ ]:
// Decision Treeの構造を表示
val dtModel = model.stages(0).asInstanceOf[org.apache.spark.ml.classification.DecisionTreeClassificationModel]

println("Decision Tree Structure:")
println(dtModel.toDebugString)

## 10. クリーンアップ

In [ ]:
// SparkSessionの停止
// spark.stop()
println("完了！")